<a href="https://colab.research.google.com/github/chtzortzi-alt/Tzortzi_Facial_Expressions/blob/main/Tzortzi_Imaging_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kagglehub

Dataset is hosted on Kaggle. The original one was last updated 6 years ago. I prefer an immediate retrieval than working locally and performing a manual upload every time I work on the assignment.

In [ ]:
import kagglehub
from pathlib import Path

dataset_path = Path(
    kagglehub.dataset_download("msambare/fer2013")
)

print("Dataset location:", dataset_path)
print("Folder contents:", [p.name for p in dataset_path.iterdir()])

Represents the dataset's location and specifies the dataset owner and dataset name. Displays the path to stored location and folder contents confirms the code's availability to access the data.

In [2]:
for item in dataset_path.iterdir():
  print(item.name, "- Directory:", item.is_dir())

NameError: name 'dataset_path' is not defined

Checks the folders' types. The original dataset contains two subsets of train and test. The training one will be further split to an 80% training and a 20% validation subset using a stratification technique to preserve the original distirbution of the seven facial expression categories. The original test set will remain unchanged.

In [ ]:
train_path = dataset_path / "train"
test_path = dataset_path / "test"

train_classes = sorted([
    folder.name
    for folder in train_path.iterdir()
    if folder.is_dir()
])

test_classes = sorted([
    folder.name
    for folder in test_path.iterdir()
    if folder.is_dir()
])

print("Training categories:", train_classes)
print("Testing categories:", test_classes)

We check the contents of each separate directory (test, train) wanting to find the same seven facial expression categories. These are sorted alphabetically: anger, disgust, fear, happy, neutral, sad and surprise. This is important so as to ensure consistency during the preprocessing, training and evaluation.

In [ ]:
from collections import Counter

extensions = Counter(
    file.suffix.lower()
    for folder in train_path.iterdir()
    if folder.is_dir()
    for file in folder.iterdir()
    if file.is_file()
)

print("Training file extensions:", extensions)

All images are in .jpg format (training set). Total number of images accounting to 28.709.

In [ ]:
from PIL import Image
import numpy as np

sample_path = sorted((train_path / "happy").glob("*.jpg"))[0]

with Image.open(sample_path) as img:
  print("Image format:", img.format)
  print("Image mode:", img.mode)
  print("Image dimensions:", img.size)

  pixels = np.array(img)

print("Array shape:", pixels.shape)
print("Pixel data type:", pixels.dtype)
print("Minimum pixel value:", pixels.min())
print("Maximum pixel value:", pixels.max())

The instance of an inspected image is grayscale. Therefore, it seems that there won't be a need for conversion. Dimensions appear to be 48x48 and the pixel value range is 0-255. The images will be normalized on a 0-1 scale.

In [ ]:
import pandas as pd

data = []

for split, path in [("train", train_path), ("test", test_path)]:
  for folder in sorted(path.iterdir()):
    if folder.is_dir():
      image_count = len([
          file for file in folder.iterdir()
          if file.suffix.lower() in [".jpg", ".jpeg", ".png"]
      ])

      data.append({
          "split": split,
          "expression": folder.name,
          "count": image_count
      })

df = pd.DataFrame(data)

display(df)

In [3]:
import matplotlib.pyplot as plt

class_counts = df.pivot(
    index = "expression",
    columns = "split",
    values = "count"
)

class_counts.plot(kind = "bar", figsize = (9, 5))

plt.title("FER2013 Class Distribution")
plt.xlabel("Facial Expression")
plt.ylabel("Number of Images")
plt.xticks(rotation = 45)
plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

In [ ]:
class_percentages = (
    class_counts.div(class_counts.sum(axis = 0), axis = 1) * 100
)

display(class_percentages.round(2))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 7, figsize = (14, 3))

for i, expression in enumerate(train_classes):
  expression_path = train_path / expression

  image_files = sorted(expression_path.glob("*.jpg"))

  image = Image.open(image_files[0])

  axes[i].imshow(image, cmap = "gray")
  axes[i].set_title(expression)
  axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
image_records = []

for expression in train_classes:
  expression_folder = train_path / expression

  for image_file in sorted(expression_folder.glob("*.jpg")):
    image_records.append({
        "file_path": str(image_file.relative_to(dataset_path)),
        "expression": expression
    })

images_df = pd.DataFrame(image_records)

display(images_df.head())
print("Total training records:", len(images_df))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

sample_df = images_df.sample(n=500, random_state=42)

mean_intensities = []
std_intensities = []

for relative_path in sample_df["file_path"]:
  image_path = dataset_path / relative_path

  with Image.open(image_path) as img:
    pixels = np.array(img, dtype = np.float32)

  mean_intensities.append(pixels.mean())
  std_intensities.append(pixels.std())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(mean_intensities, bins=30)
axes[0].set_title("Mean Pixel Intensity")
axes[0].set_xlabel("Mean Intensity (0-255)")
axes[0].set_ylabel("Number of Images")

axes[1].hist(std_intensities, bins=30)
axes[1].set_title("Pixel Intensity Standard Deviation")
axes[1].set_xlabel("Standard Deviation")
axes[1].set_ylabel("Number of Images")

plt.tight_layout()
plt.show()

In [4]:
from PIL import Image
from collections import Counter

image_properties = Counter()
file_extensions = Counter()
corrupted_files = []

for split, path in [("train", train_path), ("test", test_path)]:
  for expression in train_classes:
    folder = path / expression

    for file in sorted(folder.iterdir()):
      if file.is_file():
        file_extensions[(split, file.suffix.lower())] +=1

        try:
          with Image.open(file) as img:
            img.load()

            image_properties[
                (split, img.size, img.mode, img.format)
            ] +=1

        except Exception as e:
              corrupted_files.append((str(file), str(e)))

print("File extensions:", file_extensions)
print("Image properties:", image_properties)
print("Unreadable files:", len(corrupted_files))

if corrupted_files:
  print("Examples:", corrupted_files[:5])


NameError: name 'train_path' is not defined

Moving on to data preprocessing:

In [ ]:
combined_df = images_df.copy()

missing_files = []
mismatched_labels = []

for _, row in combined_df.iterrows():
  image_path = dataset_path / row["file_path"]
  if not image_path.is_file():
    missing_files.append(row["file_path"])
  if image_path.parent.name != row["expression"]:
    mismatched_labels.append(row["file_path"])

print("Missing files:", len(missing_files))
print("Mismatched labels:", len(mismatched_labels))
print("Total records:", len(combined_df))
print("Unique paths:", combined_df["file_path"].nunique())

In [ ]:
import hashlib
from collections import defaultdict

file_hashes = defaultdict(list)

for split_name, dataframe in [
    ("train", images_df)
]:
  for relative_path in dataframe["file_path"]:
    image_path = dataset_path / relative_path

    with open(image_path, "rb") as f:
      image_hash = hashlib.sha256(f.read()).hexdigest()

    file_hashes[image_hash].append(
        (split_name, relative_path)
    )

for expression in test_classes:
  for image_path in sorted((test_path / expression).glob("*.jpg")):

    with open(image_path, "rb") as f:
      image_hash = hashlib.sha256(f.read()).hexdigest()

    file_hashes[image_hash].append(
        ("test", str(image_path.relative_to(dataset_path)))
    )

duplicates = {
    h: records
    for h, records in file_hashes.items()
    if len(records) > 1
}

cross_split_duplicates = {
    h: records
    for h, records in duplicates.items()
    if len({split for split, path in records}) > 1
}

print("Exact duplicate groups:", len(duplicates))
print("Duplicate groups crossing splits:", len(cross_split_duplicates))

if cross_split_duplicates:
  print("\nExamples of cross-split duplicates:")
  for records in list(cross_split_duplicates.values())[:5]:
    print(records)
else:
  print("\nNo byte-identical cross-split duplicates detected.")

In [ ]:
all_labels = {}

all_labels.update(
    dict(zip(images_df["file_path"], images_df["expression"]))
)

# original test labels
for expression in test_classes:
  for path in sorted((test_path / expression).glob("*.jpg")):
    relative_path = str(path.relative_to(dataset_path))
    all_labels[relative_path] = expression

# count records of duplicat egroups
duplicate_file_count = sum(
    len(records) for records in duplicates.values()
)

cross_split_file_count = sum(
    len(records) for records in cross_split_duplicates.values()
)

# identical files with different expr. labels assigned
conflicting_groups = {}

for image_hash, records in duplicates.items():
  labels = {
      all_labels[path]
      for split, path in records
  }
  if len(labels) > 1:
    conflicting_groups[image_hash] = records

# identical files sharing by the original sets
train_test_groups = {
    image_hash: records
    for image_hash, records in duplicates.items()
    if {"train", "test"}.issubset(
      {split for split, path in records}
    )
}

print("Files belonging to duplicate groups:", duplicate_file_count)
print("Files in cross-split duplicate groups:", cross_split_file_count)
print("Groups with conflicting expression labels:", len(conflicting_groups))
print("Groups shared with the original test set:", len(train_test_groups))

print("Largest duplicate group size:", max(len(records) for records in duplicates.values()))


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

example_records = max(
    conflicting_groups.values(), key = len
)

n = min(len(example_records), 8)

fig, axes = plt.subplots(1, n, figsize=(2*n, 3))

if n == 1:
  axes = [axes]

for ax, (split, relative_path) in zip(axes, example_records[:n]):
  image_path = dataset_path / relative_path

  with Image.open(image_path) as img:
    ax.imshow(img, cmap="gray")

  ax.set_title(
      f"{split}\n{all_labels[relative_path]}", fontsize=9
    )
  ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image

image_quality_records = []

for image_path in sorted(dataset_path.rglob("*.jpg")):
  with Image.open(image_path) as img:
    pixels = np.asarray(img, dtype=np.float32)

  relative_path = image_path.relative_to(dataset_path)

  image_quality_records.append({
    "file_path": str(relative_path), "split": relative_path.parts[0],
    "expression": image_path.parent.name, "min_intensity": pixels.min(), "max_intensity": pixels.max(),
    "intensity_range": np.ptp(pixels), "std_intensity": pixels.std()
  })

quality_df = pd.DataFrame(image_quality_records)

constant_images = quality_df[
    quality_df["intensity_range"] == 0
]

low_variation_images = quality_df[
    (quality_df["intensity_range"] > 0) &
    (quality_df["std_intensity"] < 5)
]

print("Total images checked:", len(quality_df))
print("Images checked by original split:", quality_df["split"].value_counts())
print("\nconstant-intensity images:", len(constant_images))
print("Low-variation images (std < 5):", len(low_variation_images))

print("\nConstant images by  split:", constant_images["split"].value_counts())
print("Constant images by expression:", constant_images["expression"].value_counts())

print("\nConstant image examples:")
print(
    constant_images[
        ["file_path", "expression", "min_intensity", "max_intensity"]
    ].head(15).to_string(index = False)
)

assert len(quality_df) == 35887
assert (quality_df["split"] == "train").sum() == (28709)
assert (quality_df["split"] == "test").sum() == (7178)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

n = len(low_variation_images)

fig, axes = plt.subplots(1, n, figsize=(4*n, 4))

axes = np.atleast_1d(axes)

for ax, (_, row) in zip(axes, low_variation_images.iterrows()):
  image_path = dataset_path / row["file_path"]

  with Image.open(image_path) as img:
    pixels = np.asarray(img)

  ax.imshow(pixels, cmap="gray", vmin=0, vmax=255)

  ax.set_title(
      f"{row['expression']}\n"
      f"Std: {row['std_intensity']:.2f}\n"
      f"Range: {row['intensity_range']:.0f}"
  )

  ax.axis("off")

plt.tight_layout()
plt.show()

In [5]:
thresholds = [5, 10, 15, 20, 25, 30]

for threshold in thresholds:
  count = (
      (quality_df["std_intensity"] < threshold) & (quality_df["intensity_range"] > 0)
  ).sum()

  print(f"Std < {threshold}: {count} images")

NameError: name 'quality_df' is not defined

In [ ]:
candidate_images = quality_df[
    (quality_df["std_intensity"] >= 5) & (quality_df["std_intensity"] < 15)
].sort_values("std_intensity")

n = min(len(candidate_images), 20)

if n == 0:
  print("No images found in this range")

else:
  fig, axes = plt.subplots(
      nrows=(n+4)//5, ncols=5, figsize=(12, 3*((n+4)//5))
  )

  axes = np.asarray(axes).flatten()

  for ax, (_, row) in zip(axes, candidate_images.head(n).iterrows()):
    image_path = dataset_path / row["file_path"]

    with Image.open(image_path) as img:
      pixels = np.asarray(img)

    ax.imshow(pixels, cmap="gray", vmin=0, vmax=255)

    ax.set_title(
      f"{row['expression']} | Std: {row['std_intensity']:.2f} | Range: {row['intensity_range']:.0f}",
      fontsize=9
    )

    ax.axis("off")

  for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

decision to exclude all images with 0 <= std < 5

In [ ]:
clean_df = quality_df.loc[
    quality_df["split"] == "train", ["file_path", "expression"]
].copy()

print("Original training images:", len(clean_df))

hash_lookup = {
    path: image_hash
    for image_hash, records in file_hashes.items()
    for split, path in records
}

clean_df["hash"] = clean_df["file_path"].map(hash_lookup)

assert clean_df["hash"].notna().all()

# constant intensity images (min=max) and images with std < 5
unusable_paths = set(
    quality_df.loc[
        (quality_df["intensity_range"] == 0) |
        (
            (quality_df["intensity_range"] > 0) &
            (quality_df["std_intensity"] < 5)
        ),
        "file_path"
    ]
)

before = len(clean_df)
clean_df = clean_df[
    ~clean_df["file_path"].isin(unusable_paths)
].copy()

print("Removed unusable images:", before - len(clean_df))

# groups with conflicting labels
conflicting_hashes = set(conflicting_groups.keys())

before = len(clean_df)
clean_df = clean_df[
    ~clean_df["hash"].isin(conflicting_hashes)
].copy()

print("Removed conflicting-label recordings:", before - len(clean_df))

# images identical to original test set images
test_hashes = {
    image_hash
    for image_hash, records in file_hashes.items()
    if any(split == "test" for split, path in records)
}

before = len(clean_df)
clean_df = clean_df[
    ~clean_df["hash"].isin(test_hashes)
].copy()

print("Removed test-overlapping records:", before - len(clean_df))

# keep one copy per remaining hash
before = len(clean_df)

clean_df = (
    clean_df
    .sort_values("file_path")
    .drop_duplicates(subset="hash", keep="first")
    .reset_index(drop=True)
)

print("Removed remaining duplicate copies:", before - len(clean_df))
print("Final cleaned training pool:", len(clean_df))
print("Class distribution:", clean_df["expression"].value_counts())

assert clean_df["hash"].is_unique
assert not set(clean_df["hash"]) & test_hashes

In [ ]:
from sklearn.model_selection import train_test_split

train_clean_df, val_clean_df = train_test_split(
    clean_df, test_size=0.20, random_state=42, stratify=clean_df["expression"]
)

train_clean_df = train_clean_df.reset_index(drop=True)
val_clean_df = val_clean_df.reset_index(drop=True)

print("Cleaned training images:", len(train_clean_df))
print("Cleaned validation images:", len(val_clean_df))

assert not (
    set(train_clean_df["hash"]) & set(val_clean_df["hash"])
)

In [ ]:
print("Training class distribution:", train_clean_df["expression"].value_counts())
print("Validation class distribution:", val_clean_df["expression"].value_counts())

assert train_clean_df["file_path"].is_unique
assert val_clean_df["file_path"].is_unique
assert train_clean_df["hash"].is_unique
assert val_clean_df["hash"].is_unique

assert not (
    set(train_clean_df["hash"]) & set(val_clean_df["hash"])
)

assert not (
    set(train_clean_df["hash"]) | set(val_clean_df["hash"])
) & test_hashes

print("All cleaned-split integrity checks passed.")

In [ ]:
split_percentages = pd.DataFrame({
    "training (%)": (train_clean_df["expression"].value_counts(normalize=True)*100),
    "validation (%)": (val_clean_df["expression"].value_counts(normalize=True)*100)
}).sort_index()

display(split_percentages.round(2))

split_percentages.plot(
    kind="bar", figsize=(10, 5)
)

plt.title("Class distribution after stratified split")
plt.xlabel("Facial expression")
plt.ylabel("Percentage of images (%)")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

output_dir = Path("/content/cleaned_splits")
output_dir.mkdir(parents = True, exist_ok = True)

train_clean_df[["file_path", "expression"]].to_csv(output_dir / "train_clean_split.csv", index = False)
val_clean_df[["file_path", "expression"]].to_csv(output_dir / "validation_clean_split.csv", index = False)

print("Clean split files saved to:", output_dir / "train_clean_split.csv", output_dir / "validation_clean_split.csv")

https://raw.githubusercontent.com/chtzortzi-alt/Tzortzi_Facial_Expressions/refs/heads/main/final_cleaned_splits/train_clean_split.csv
https://raw.githubusercontent.com/chtzortzi-alt/Tzortzi_Facial_Expressions/refs/heads/main/final_cleaned_splits/validation_clean_split.csv

In [6]:
train_url = "https://raw.githubusercontent.com/chtzortzi-alt/Tzortzi_Facial_Expressions/refs/heads/main/final_cleaned_splits/train_clean_split.csv"
val_url = "https://raw.githubusercontent.com/chtzortzi-alt/Tzortzi_Facial_Expressions/refs/heads/main/final_cleaned_splits/validation_clean_split.csv"

train_clean_df = pd.read_csv(train_url)
val_clean_df = pd.read_csv(val_url)

print("Training images:", len(train_clean_df))
print("Validation images:", len(val_clean_df))

NameError: name 'pd' is not defined

label encoding

In [ ]:
label_mapping = {
    expression: i
    for i, expression in enumerate(train_classes)
}

print("Label mapping:", label_mapping)

train_clean_df["label"] = train_clean_df["expression"].map(label_mapping).astype("int32")
val_clean_df["label"] = val_clean_df["expression"].map(label_mapping).astype("int32")

print("Training label type:", train_clean_df["label"].dtype)
print("Validation label type:", val_clean_df["label"].dtype)

display(train_clean_df.head())

In [ ]:
print("Training labels:")
print(sorted(train_clean_df["label"].unique()))
print("Validation labels:")
print(sorted(val_clean_df["label"].unique()))

print("\nMissing training labels:", train_clean_df["label"].isna().sum())
print("Missing validation labels:", val_clean_df["label"].isna().sum())

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

In [ ]:
def load_image(file_path, label):
  image = tf.io.read_file(file_path)

  image = tf.image.decode_jpeg(
      image, channels = 1
  )

  image = tf.image.convert_image_dtype(
      image, tf.float32
  )

  label = tf.cast(label, tf.int32)

  return image, label

^ normalizes

In [ ]:
sample_path = str(
    dataset_path / train_clean_df.iloc[0]["file_path"]
)

sample_label = int(train_clean_df.iloc[0]["label"])

image_tensor, label_tensor = load_image(
    sample_path, sample_label
)

print("Tensor shape:", image_tensor.shape)
print("Tensor data type:", image_tensor.dtype)
print("Label data type:", label_tensor.dtype)

print("Minimum pixel value:", float(tf.reduce_min(image_tensor)))
print("Maximum pixel value:", float(tf.reduce_max(image_tensor)))

print("Numerical label:", int(label_tensor.numpy()))
print("Expression:", train_clean_df.iloc[0]["expression"])

In [ ]:
train_paths = [
    str(dataset_path / path)
    for path in train_clean_df["file_path"]
]

val_paths = [
    str(dataset_path / path)
    for path in val_clean_df["file_path"]
]

train_labels = train_clean_df["label"].to_numpy()
val_labels = val_clean_df["label"].to_numpy()

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_paths, train_labels)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_paths, val_labels)
)

print("Training records:", train_ds.cardinality().numpy())
print("Validation records:", val_ds.cardinality().numpy())

In [ ]:
autotune = tf.data.AUTOTUNE
batch_size = 32

train_ds = train_ds.shuffle(
    buffer_size = len(train_paths), seed = 42
)

train_ds = train_ds.map(
    load_image, num_parallel_calls = autotune
)

val_ds = val_ds.map(
    load_image, num_parallel_calls = autotune
)

train_ds = train_ds.batch(batch_size)
val_ds = val_ds.batch(batch_size)

train_ds = train_ds.prefetch(autotune)
val_ds = val_ds.prefetch(autotune)

print("Training batches:", train_ds.cardinality().numpy())
print("Validation batches:", val_ds.cardinality().numpy())

In [7]:
test_records = []

for expression in test_classes:
  expression_folder = test_path / expression

  for image_path in sorted(expression_folder.glob("*.jpg")):
    test_records.append({
        "file_path": str(image_path.relative_to(dataset_path)), "expression": expression
    })

test_df = pd.DataFrame(test_records)

test_df["label"] = (
    test_df["expression"]
    .map(label_mapping)
    .astype("int32")
)

test_paths = [
    str(dataset_path / path)
    for path in test_df["file_path"]
]

test_labels = test_df["label"].to_numpy()

test_ds = tf.data.Dataset.from_tensor_slices(
    (test_paths, test_labels)
)

test_ds = (
    test_ds
    .map(load_image, num_parallel_calls = autotune)
    .batch(batch_size)
    .prefetch(autotune)
)

print("Original test images:", len(test_df))
print("Test batches:", test_ds.cardinality().numpy())

NameError: name 'test_classes' is not defined

In [ ]:
for name, dataset in [("Training", train_ds), ("Validation", val_ds), ("Test", test_ds)]:
  for images, labels in dataset.take(1):
    print(f"{name} dataset")
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Image dtype:", images.dtype)
    print("Label dtype:", labels.dtype)
    print("Minimum pixel:", tf.reduce_min(images).numpy())
    print("Maximum pixel:", tf.reduce_max(images).numpy())

    assert images.shape[1:] == (48, 48, 1)
    assert images.dtype == tf.float32
    assert labels.dtype == tf.int32
    assert tf.reduce_min(images).numpy() >= 0
    assert tf.reduce_max(images).numpy() <= 1

print("\nAll TensorFlow dataset checks passed.")

ready for model selection and training

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

mlp = keras.Sequential([
    layers.Input(shape=(48, 48, 1)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(7, activation="softmax")
])

mlp.summary()

In [ ]:
mlp.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)

MLP training with ideas derived from the Smart Sprinkler exercise using Keras and image data

In [ ]:
epochs = 10

mlp_history = mlp.fit(
    train_ds, validation_data = val_ds, epochs = epochs, verbose = 1
)

In [ ]:
history = mlp_history.history

epoch_numbers = range(1, len(history["loss"]) + 1)

plt.figure(figsize=(8, 5))

plt.plot(epoch_numbers, history["loss"], label="Training loss")
plt.plot(epoch_numbers, history["val_loss"], label="Validation loss")

plt.title("MLP training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))

plt.plot(epoch_numbers, history["accuracy"], label="Training accuracy")
plt.plot(epoch_numbers, history["val_accuracy"], label="Validation accuracy")

plt.title("MLP training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
mlp_val_loss, mlp_val_acc = mlp.evaluate(val_ds, verbose=1)

print("MLP validation loss:", round(mlp_val_loss, 4))
print("MLP validation accuracy:", round(mlp_val_acc, 4))

In [1]:
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay)

mlp_probabilities = mlp.predict(val_ds, verbose=1)
mlp_predictions = np.argmax(mlp_probabilities, axis=1)

true_labels = np.concatenate([labels.numpy() for images, labels in val_ds])

print("Predictions:", len(mlp_predictions))
print("True labels:", len(true_labels))

print(classification_report(true_labels, mlp_predictions, labels=list(range(7)), target_names=train_classes, zero_division=0))

cm = confusion_matrix(true_labels, mlp_predictions, labels=list(range(7))

gif, ax = plt.subplots(figsize=(9, 7))

ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_classes).plot(ax=ax, cmap="Blues", xticks_rotation=45, values_format="d")

plt.title("MLP validation confusion matrix")
plt.tight_layout()
plt.show()

SyntaxError: '(' was never closed (3556835947.py, line 13)